In [1]:
# ============================================================
# INGEST DRIVERS — LOCAL EXECUTION
# ============================================================

import os
from pyspark.sql.types import StructType, StructField, StringType, DateType
import nbformat
from nbconvert import PythonExporter

In [2]:
# -----------------------------------------
# 1. Load environment + helpers
# -----------------------------------------
def run_notebook(path):
    with open(path, "r") as f:
        nb = nbformat.read(f, as_version=4)
    source, _ = PythonExporter().from_notebook_node(nb)
    exec(source, globals())

run_notebook("/Users/manoharazalki/F1-Analytics/notebooks/00-common/01_environment_config.ipynb")
run_notebook("/Users/manoharazalki/F1-Analytics/notebooks/00-common/02_bronze_helpers.ipynb")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/09 13:15:04 WARN Utils: Your hostname, Manohars-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 10.68.78.178 instead (on interface en0)
26/06/09 13:15:04 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/09 13:15:04 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/09 13:15:04 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/06/09 13:15:04 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


===== F1 Analytics Environment Configuration =====
BASE_PATH:        /Users/manoharazalki/F1-Analytics/data
LANDING_PATH:     /Users/manoharazalki/F1-Analytics/data/landing
BRONZE_PATH:      /Users/manoharazalki/F1-Analytics/data/bronze
SILVER_PATH:      /Users/manoharazalki/F1-Analytics/data/silver
GOLD_PATH:        /Users/manoharazalki/F1-Analytics/data/gold
INCREMENTAL_PATH: /Users/manoharazalki/F1-Analytics/data/incremental
CONTROL_PATH:     /Users/manoharazalki/F1-Analytics/data/control


In [3]:
# -----------------------------------------
# 2. Detect latest landing batch folder
# -----------------------------------------
batch_folders = sorted([
    f for f in os.listdir(LANDING_PATH)
    if os.path.isdir(os.path.join(LANDING_PATH, f))
])

if not batch_folders:
    raise Exception("❌ No batch folders found in landing!")

p_batch_id = batch_folders[-1]
print("Detected landing batch folder:", p_batch_id)

BATCH_LANDING_PATH = f"{LANDING_PATH}/{p_batch_id}"

Detected landing batch folder: 2025-01


In [4]:
# -----------------------------------------
# 3. Generate pipeline batch_id
# -----------------------------------------
batch_id = generate_batch_id()
print("Pipeline batch_id:", batch_id)

Pipeline batch_id: 20260609_131505


In [5]:
# -----------------------------------------
# 4. Define source + target paths
# -----------------------------------------
source_file = f"{BATCH_LANDING_PATH}/drivers.json"
target_path = f"{BRONZE_PATH}/drivers"
os.makedirs(target_path, exist_ok=True)

print("Source file:", source_file)
print("Target path:", target_path)

Source file: /Users/manoharazalki/F1-Analytics/data/landing/2025-01/drivers.json
Target path: /Users/manoharazalki/F1-Analytics/data/bronze/drivers


In [6]:
# -----------------------------------------
# 5. Define schema (nested structure)
# -----------------------------------------
name_schema = StructType([
    StructField("givenName", StringType(), True),
    StructField("familyName", StringType(), True),
])

drivers_schema = StructType([
    StructField("driverId", StringType(), True),
    StructField("name", name_schema, True),
    StructField("dateOfBirth", DateType(), True),
    StructField("nationality", StringType(), True),
    StructField("url", StringType(), True)
])

In [7]:
# -----------------------------------------
# 6. Read JSON
# -----------------------------------------
drivers_df = (
    spark.read
        .format("json")
        .schema(drivers_schema)
        .option("mode", "FAILFAST")
        .load(source_file)
)

print("✔ Drivers file read successfully")
drivers_df.show(5, truncate=False)

✔ Drivers file read successfully
+---------+-------------------+-----------+-----------+----------------------------------------------+
|driverId |name               |dateOfBirth|nationality|url                                           |
+---------+-------------------+-----------+-----------+----------------------------------------------+
|abate    |{carlo, abate}     |1932-07-10 |italian    |http://en.wikipedia.org/wiki/Carlo_Mario_Abate|
|abecassis|{george, abecassis}|1913-03-21 |british    |http://en.wikipedia.org/wiki/George_Abecassis |
|acheson  |{kenny, acheson}   |1957-11-27 |british    |http://en.wikipedia.org/wiki/Kenny_Acheson    |
|adams    |{philippe, adams}  |1969-11-19 |belgian    |http://en.wikipedia.org/wiki/Philippe_Adams   |
|ader     |{walt, ader}       |1913-12-15 |american   |http://en.wikipedia.org/wiki/Walt_Ader        |
+---------+-------------------+-----------+-----------+----------------------------------------------+
only showing top 5 rows


In [8]:
# -----------------------------------------
# 7. Add ingestion metadata
# -----------------------------------------
drivers_final_df = add_ingestion_metadata(drivers_df, source_file)

print("✔ Metadata added")
drivers_final_df.show(5, truncate=False)

✔ Metadata added
+---------+-------------------+-----------+-----------+----------------------------------------------+--------------------------+-------------------------------------------------------------------+
|driverId |name               |dateOfBirth|nationality|url                                           |ingest_timestamp          |source_file                                                        |
+---------+-------------------+-----------+-----------+----------------------------------------------+--------------------------+-------------------------------------------------------------------+
|abate    |{carlo, abate}     |1932-07-10 |italian    |http://en.wikipedia.org/wiki/Carlo_Mario_Abate|2026-06-09 13:15:06.976185|/Users/manoharazalki/F1-Analytics/data/landing/2025-01/drivers.json|
|abecassis|{george, abecassis}|1913-03-21 |british    |http://en.wikipedia.org/wiki/George_Abecassis |2026-06-09 13:15:06.976185|/Users/manoharazalki/F1-Analytics/data/landing/2025-01/drivers

In [9]:
# -----------------------------------------
# 8. Write to Bronze (partitioned by batch_id)
# -----------------------------------------
write_to_bronze(
    drivers_final_df,
    f"{target_path}/data.parquet",
    batch_id
)

print(f"✔ Drivers Bronze written to: {target_path}/data.parquet")

✔ Drivers Bronze written to: /Users/manoharazalki/F1-Analytics/data/bronze/drivers/data.parquet


In [10]:
# -----------------------------------------
# 9. Read back for validation
# -----------------------------------------
spark.read.parquet(f"{target_path}/data.parquet").show(5, truncate=False)

+---------+-------------------+-----------+-----------+----------------------------------------------+--------------------------+-------------------------------------------------------------------+---------------+
|driverId |name               |dateOfBirth|nationality|url                                           |ingest_timestamp          |source_file                                                        |batch_id       |
+---------+-------------------+-----------+-----------+----------------------------------------------+--------------------------+-------------------------------------------------------------------+---------------+
|abate    |{carlo, abate}     |1932-07-10 |italian    |http://en.wikipedia.org/wiki/Carlo_Mario_Abate|2026-06-09 13:15:07.060106|/Users/manoharazalki/F1-Analytics/data/landing/2025-01/drivers.json|20260609_131505|
|abecassis|{george, abecassis}|1913-03-21 |british    |http://en.wikipedia.org/wiki/George_Abecassis |2026-06-09 13:15:07.060106|/Users/manohara